# Exploring Geopandas
This notebook is an exploration of crime statistics from 2024 in the state of South Carolina.

---

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

## Constants
Pre-declaring my data source locations just makes it easier for me to get what I want (raw data) and put things where I want as I work with my data (output images)

In [ ]:
# Project root relative to this notebook.
PROJECT_ROOT = Path.cwd().parent

# Location for output artifacts.
OUTPUT_DIR = PROJECT_ROOT / 'output'

# Location of my raw data
DATA_DIR = PROJECT_ROOT / 'data' 

from db_config import DB_URL

# TIGER/Line shapefiles for all US counties in the US.
COUNTY_GEO_DATA = DATA_DIR / 'tl_2025_us_county.zip'

# 5-year American Community Survey population data.
COUNTY_POPULATION = DATA_DIR / 'ACSDP5Y2024.DP05_2026-07-21T172204/ACSDP5Y2024.DP05-Data.csv'

## Data Engineering.
No single data source had everything I wanted. This series of cells is to collect data, normalize headings, and merge them together, then save them to their own database tables when applicable. The tables will serve as a "cache" of target data that will be used in a final dashboard.

In [ ]:
import geopandas as gpd
gdf = gpd.read_file(
    COUNTY_GEO_DATA,
    columns=['STATEFP', 'GEOID', 'GEOIDFQ', 'NAME', 'geometry'],
)

In [ ]:
sc = gdf[gdf['STATEFP'] == "45"].to_crs(4326)

del sc['STATEFP']
sc = sc.rename(columns={
    "GEOID": "fips",
    "GEOIDFQ": "state_county_id",
    "NAME": "county_name"
})

In [ ]:
import pandas as pd
county_populations = pd.read_csv(
    COUNTY_POPULATION,
    skiprows=[1],
    usecols=["GEO_ID", "DP05_0001E", "DP05_0002E", "DP05_0003E"],
    dtype={
        "DP05_0001E": "int",
        "DP05_0002E": "int",
        "DP05_0003E": "int"
    }
)

In [ ]:
county_populations = county_populations.rename(columns={
    "GEO_ID": "state_county_id",
    "DP05_0001E": "total_population",
    "DP05_0002E": "total_male_population",
    "DP05_0003E": "total_female_population"
})

In [ ]:
sc_county_geoid = sc['state_county_id'].tolist()
sc_county_geoid.append('Geography')
sc_county_filter = county_populations['state_county_id'].isin(sc_county_geoid)
sc_county_populations = county_populations[sc_county_filter]
sc_with_pop_estimates = pd.merge(sc, sc_county_populations, on="state_county_id", how="left")
sc_with_pop_estimates = sc_with_pop_estimates.reindex(columns=[
    "county_name",
    "fips",
    "state_county_id",
    "total_population",
    "total_male_population",
    "total_female_population",
    "geometry"])

### "Cache" Merged County Data.
Storing relevant county geometry and the total populations of each county, plus totals for male and female populations.

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.types import Integer, String

engine = create_engine(DB_URL)
with engine.connect() as connection:
    sc_with_pop_estimates.to_postgis(
        'sc_counties', 
        con=connection, 
        if_exists='replace',
        dtype={
            "state_county_id": String(),
            "total_population": Integer(),
            "total_male_population": Integer(),
            "total_female_population": Integer()
        }
    )

In [ ]:
import plotly.express as px
import plotly.graph_objects as go


> If something is working as expect in your plot check the dtype of your column. Below I could not get the color
> scale of `viridis` to apply. Turns out the column was dtype of `string` not `int`

#### Notes
Counties colored with the bottom of the color scale have a population of -1 because those counties were omitted from the report to the census bureau.

In [ ]:
geojson = sc_with_pop_estimates.__geo_interface__

minx, miny, maxx, maxy = sc_with_pop_estimates.total_bounds

center = {"lat": (miny + maxy) / 2, "lon": (minx + maxx) / 2 }

fig = px.choropleth_map(
    sc_with_pop_estimates,
    geojson=geojson,
    locations=sc_with_pop_estimates.index,
    color_continuous_scale='viridis_r',
    color='total_population',
    zoom=6,
    opacity=0.75,
    center=center,
    hover_name="county_name",
    hover_data={'total_population': ':,'}    # thousands separator in tooltip
)

fig.update_layout(
    width=700,
    height=700,
    margin={"r": 0, "t": 0, "l": 0, "b": 0},# removes the default white padding
    dragmode=False,
    xaxis=dict(fixedrange=True),
    yaxis=dict(fixedrange=True)
)

fig.show(config={
    "scrollZoom": False
})

In [ ]:
total_crimes_query = """
SELECT s.county_name,
CASE
when GROUPING(td.offender_sex) = 0 THEN 'sex'
WHEN GROUPING(td.location) = 0 THEN 'location'
ELSE 'total'
END AS dimension,
COALESCE(td.offender_sex, td.location, td.offense_type, 'ALL') as category,
COUNT(DISTINCT td.incident_id) as incident_count
FROM sc_county_law_info s
JOIN target_data td ON td.fips = s.fips
GROUP BY GROUPING SETS(
(s.county_name),
(s.county_name, td.offender_sex),
(s.county_name, td.location),
(s.county_name, td.offense_type)
)
ORDER BY s.county_name, dimension, incident_count desc
"""
with engine.connect() as connection:
    result = pd.read_sql(total_crimes_query, con=connection)

In [ ]:
result_pivot = result.pivot_table(index="county_name", columns=["dimension", "category"],
                                  values="incident_count", fill_value=0)
sc_with_pop_estimates = sc_with_pop_estimates.set_index('county_name')

In [ ]:
# sc_with_pop_estimates['total_incident_rate_per_10k'] = result_pivot[('total', 'ALL')].div(sc_with_pop_estimates['total_population'], axis=0) * 10000

## Pause
At this point in the process I have started to realize that, for the sake of completeing this project, I need to narrow scope and focus on just and handful of data points. The questions is: 
- Which ones?
- And how many to start with?

#### Targets
- Crime rates per crime type (per 10k residents)
- Crime rates per location (per 10k residents)
- Top 5 type category
- Top 5 location cateogry
- Pie % type (9 + "other")
- Total rate of crime diff with surrounding counties.

In [ ]:
percentage_of_incident_type = result_pivot['total'].div(result_pivot[('total', 'ALL')], axis=0).copy()
perc_of_type_at_york = percentage_of_incident_type.loc['York'].copy()

In [ ]:
perc_of_type_at_york_sorted = perc_of_type_at_york.sort_values(ascending=False)

END = 10

top_9_type_at_york = perc_of_type_at_york_sorted.iloc[1:END].copy()
top_9_type_at_york['Other'] = perc_of_type_at_york_sorted.iloc[END:].sum()

top_9_type_at_york.index

top_type_fig = px.bar(
    top_9_type_at_york.sort_values(),
    orientation='h',
    title="Top Crime Types in York County, SC",
    subtitle='Top 10 types of crime in York County, SC'
)

top_type_fig.update_layout(
    xaxis_tickformat='.0%',
    showlegend=False
)

top_type_fig.show(config={
    'toImageButtonOptions': {
        'format': 'png',
        'filename': 'top_type_in_york'
    }
})

In [ ]:
perc_of_incident_location = result_pivot['location'].div(result_pivot[('total', 'ALL')], axis=0).copy()
perc_of_location_at_york = perc_of_incident_location.loc['York'].copy()

perc_of_location_at_york_sorted = perc_of_location_at_york.sort_values(ascending=False)

top_9_location_at_york = perc_of_location_at_york_sorted.iloc[:END].copy()

top_9_location_at_york['Other'] = perc_of_location_at_york_sorted.iloc[END:].sum()


top_location_fig = px.bar(
    top_9_location_at_york.sort_values(),
    orientation='h',
    title='Top Locations of Incidents',
    subtitle='The top 10 locations that incidents occur at in York County, SC',
)

top_location_fig.update_layout(
    xaxis_tickformat='.0%',
    showlegend=False
)

top_location_fig.show(config={
    'toImageButtonOptions': {
        'format': 'png',
        'filename': 'top_location_in_york'
    }
})

In [ ]:
def get_surrounding_counties(county: str) -> gpd.Geopandas:
    query = text("""
    SELECT
        neighbor.county_name, 
        neighbor.geometry
    FROM sc_counties AS neighbor
    JOIN sc_counties AS target ON target.county_name = :county
    WHERE target.county_name = :county
      AND ST_DWithin(
          ST_Transform(neighbor.geometry, 3360), 
          ST_Transform(target.geometry, 3360), 
          0.001
        )   
    """)
    with engine.connect() as connection:
        nearest = gpd.read_postgis(query, con=connection, geom_col="geometry", params={"county": county})
        return nearest

In [ ]:
surrounding_york = get_surrounding_counties('York')

In [ ]:
around_york = surrounding_york['county_name'].tolist()
york_filter = sc_with_pop_estimates.index.isin(around_york)

In [ ]:
# sc_with_pop_estimates = sc_with_pop_estimates.reset_index()

not_around_york = sc_with_pop_estimates.loc[~york_filter]
is_around_york = sc_with_pop_estimates.loc[york_filter]

geojson = not_around_york.__geo_interface__

minx, miny, maxx, maxy = sc_with_pop_estimates.total_bounds

center = {"lat": (miny + maxy) / 2, "lon": (minx + maxx) / 2 }

fig = px.choropleth_map(
    not_around_york,
    geojson=geojson,
    locations=not_around_york.index,
    # color_continuous_scale='greys',
    color_discrete_sequence=['gray'],
    zoom=6,
    opacity=0.75,
    center=center,
    hover_data={'total_population': ':,'}    # thousands separator in tooltip
)

york_geojson = is_around_york.__geo_interface__

fig.add_trace(
    go.Choroplethmap(
        geojson=york_geojson,
        locations=is_around_york.index,
        z=is_around_york['total_population'],
        colorscale='OrRd',
    )
)

fig.update_layout(
    width=700,
    height=700,
    margin={"r": 0, "t": 0, "l": 0, "b": 0},# removes the default white padding
    dragmode=False,
    xaxis=dict(fixedrange=True),
    yaxis=dict(fixedrange=True)
)

fig.show(config={
    "scrollZoom": False
})

#### Comparing Surrounding Counties
I am not positive I will need this for my MVP of this project. But I do want to be able to compare surrouding counties at some point. Additionally the below query can also be performed from the Geopandas data frame. I had an example of that in a previous example of this notebook.

#### Omit
I will be omitting this percent_of_sex dataframe for the time being due to inaccurate percent calculation. This is due to the fact that I am grabbing incidents by offender, and an incident can have more than one offender. I will revisit.

In [ ]:
crime_rates_per_type_county_per_10k_residents = result_pivot['total'].div(sc_with_pop_estimates['total_population'], axis=0) * 10000

In [ ]:
clrs = crime_rates_per_type_county_per_10k_residents.reset_index()
merged_type = pd.merge(sc_with_pop_estimates, clrs, on="county_name")

#### Using Geopandas to Find Surrounding Counties
Here is an example of how to use Geopandas to find surrounding counties. Instead of the earlier Postgis method.

```python
met = merged_type.to_crs(epsg=3360)

york_row = met[met['county_name'] == 'Richland']
b = york_row.geometry.buffer(500).union_all()
m = met.geometry.intersects(b)
ax = merged_type[m].plot(
    cmap="viridis",
    column="Aggravated Assault",
    legend=True
)
ax.axis('off')
plt.show()
```

In [ ]:
indexed_by_name = merged_type.set_index('county_name')

In [ ]:
indexed_by_name.loc[:, 'ALL':].sort_values(ascending=True, by="ALL")['ALL'].head(12).plot(kind="barh")
plt.savefig(OUTPUT_DIR / "bottom_twelve.png", dpi=150)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))

sns.heatmap(
    (percentage_of_incident_type * 100).iloc[:, 1:],
    cmap='YlOrRd',
    # annot=True,
    fmt='.1f',
    linewidths=0.5
)

plt.title("Crime Location Rates per 10,000 Residents by County and Type")
plt.xlabel('Location of Crime')
plt.ylabel('SC County')
plt.tight_layout()

note = (
    "Note: 'Rate per 10,000 residents' allows fair comparison across counties of different sizes "
    "by showing how many crimes occurred for every 10,000 people. "
    "The 'Residence/Home' category is often highest because homes are where people spend most of "
    "their time, store valuable property, and are locations where offenders know patterns of "
    "occupancy. This doesn't mean homes are inherently dangerous — "
    "it reflects opportunity (routine activity theory: a motivated offender + a suitable target "
    "without capable guardianship)."
)

plt.figtext(0.5, -0.08, note, ha='center', fontsize=9, style='italic', color='#555555', wrap=True)

plt.savefig(OUTPUT_DIR / "crime_location_per_county_corrected.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# perc_of_incident_location.sum(axis=1)
row_totals = result_pivot['location'].sum(axis=1)
pivot_pct = result_pivot['location'].div(row_totals, axis=0)

In [ ]:
top_locations = result_pivot['location'].sum(axis=0).nlargest(10).index.tolist()
pivot_top = pivot_pct[top_locations].copy()
pivot_top['OTHER'] = 1.0 - pivot_top.sum(axis=1)

In [ ]:
target = surrounding_york['county_name'].tolist()

In [ ]:
plt.figure(figsize=(12, 8))

sns.heatmap(
    pivot_top.loc[target],
    cmap='YlGnBu',
    annot=True,
    fmt='.1%',
    linewidths=0.5,
    cbar_kws={'label': '% of County Total Crime'}
)

plt.title("Crime Rates per 10,000 Residents by County and Type")
plt.xlabel('Type of Crime')
plt.ylabel('County')

plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.savefig(OUTPUT_DIR / 'crime_location_surrounding_york.png', bbox_inches='tight', dpi=150)
plt.show()

#### Notes
> I am moving away from agencies for the time being, in many cases a given fips / county can have 1 or more agencies, so it causes issues for me when trying to aggregate data on postgis.
> 
> Instead I will focus on aggregating results for each seperate context: by county and by department.
>
> However, this throws a wrench in my original plan to use the ORI of the department to map a incident to a particular county.

In [ ]:
# sc_pop_cleaned = sc_with_pop_estimates[['GEOIDFQ', 'GEOID', 'NAME', 'geometry']]
# sc_pop_cleaned.columns = ['geo_fips', 'fips', 'county_name', 'geometry']
# sc_pop_cleaned['fips'] = sc_pop_cleaned['fips'].astype('int64')
# sc_pop_cleaned.to_postgis('sc_county_law_info', con=engine, index=False, if_exists='replace')
# result = sc_pop_cleaned.join(sc_law_cleaned, how="inner", on="fips", lsuffix='_l', rsuffix='_r')
# result
# sc_county_info = pd.merge(sc_pop_cleaned, sc_law_cleaned, on='fips', how="inner")
# sc_county_info
# sc_county_info.to_postgis('sc_county_law_info', con=engine, index=False, if_exists='replace')

#### Notes
> The new plan is to grab the fips of a given incident nad merge that to the target_data. That will be the new merge point. Current question is how?
>
> Gemini can in handy here:

```sql
ALTER TABLE target_data ADD COLUMN fips bigint;

UPDATE target_data td
    SET fips = cw.fips
    FROM sc_law_enforcment_crosswalk cw
    WHERE td.ori = cw.ori; 
```
> I did not know about this.

In [ ]:
target_sql = """
SELECT 
    i.incident_id as incident_id,
    a.ori as ori,
    lt.location_name as location,
    off.age_num as offender_age, 
    rr.race_desc as offender_race, 
    off.sex_code as offender_sex, 
    a.ncic_agency_name as arresting_agency, 
    ot.offense_name as offense_type
FROM nibrs_offense o
    JOIN nibrs_incident i using (incident_id)
    JOIN nibrs_offense_type ot using (offense_code)
    JOIN agencies a using (agency_id)
    JOIN nibrs_offender off using (incident_id)
    JOIN ref_race rr using (race_id)
    JOIN nibrs_location_type lt using (location_id);
"""

with engine.connect() as connection:
    target_data = pd.read_sql(sql=target_sql, con=connection)

# sc_with_pop_estimates = pd.merge(sc, total_pop_per_county, on="GEOIDFQ", how="left")
final_boss = pd.merge(sc_county_info, target_data, on='ori', how='right')

In [ ]:
# with engine.connect() as connection:
#     final_boss.to_postgis(
#         'sc_crimes_by_county', 
#         con=connection, 
#         if_exists='replace',
#         index=False,
#         chunksize=25000,
#     )

In [ ]:
engine.dispose()

## Data Sources
- [DP05](https://data.census.gov/table/ACSDP1Y2024.DP05?g=040XX00US45)
- [DP05](https://data.census.gov/table/ACSDP5Y2024.DP05?q=South+Carolina+Populations+of+Counties)
- [county](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.2024.html#list-tab-790442341)
- [crosswalk](https://www.icpsr.umich.edu/web/NACJD/studies/35158/versions/V2)

## Spatial Autocorrelation — LISA & Moran's I
Global and local measures of spatial autocorrelation for the **total crime rate per 10,000 residents** across SC counties (2024 NIBRS).

- **Moran's I** (global): the overall degree of spatial clustering.
- **LISA** (local indicator of spatial association): which counties form high-high / low-low / high-low / low-high clusters.

These cells are self-contained: they re-read everything fresh from the database so they don't depend on the (fragile) state of the variables above.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import libpysal
from libpysal.weights import Queen, lag_spatial
import esda


In [ ]:
# 1) Load county geometry + population, and per-county incident counts.
sc_crime = gpd.read_postgis(
    "SELECT county_name, total_population, geometry FROM sc_counties",
    con=engine,
    geom_col="geometry",
).reset_index(drop=True)

incident_counts = pd.read_sql(
    text("""
        SELECT s.county_name, COUNT(DISTINCT td.incident_id) AS incidents
        FROM target_data td
        JOIN sc_county_law_info s ON s.fips = td.fips
        GROUP BY s.county_name
    """),
    con=engine,
)

sc_crime = sc_crime.merge(incident_counts, on="county_name", how="left")
sc_crime["incidents"] = sc_crime["incidents"].fillna(0).astype(int)
sc_crime["rate_per_10k"] = sc_crime["incidents"] / sc_crime["total_population"] * 10000
sc_crime[['county_name', 'total_population', 'incidents', 'rate_per_10k']]

In [ ]:
# 2) Spatial weights: Queen contiguity (share a border/corner).
w = Queen.from_dataframe(sc_crime, use_index=True)
print("Islands (counties with no neighbor):", w.islands)
print("Connected components:", w.n_components)

In [ ]:
# 3) Global Moran's I.
y = sc_crime["rate_per_10k"].values

moran = esda.Moran(y, w)

print("Moran's I:", round(moran.I, 4))
print("Expected I under randomization:", round(moran.EI, 4))
print("z-score:", round(moran.z_norm, 3))
print("p-value:", round(moran.p_norm, 4))

In [ ]:
# 4) Local Moran (LISA): quadrant + significance per county.
lisa = esda.Moran_Local(y, w, permutations=999, seed=42)

sc_crime["lisa_I"] = lisa.Is
sc_crime["lisa_p"] = lisa.p_sim
sc_crime["lisa_q"] = lisa.q

quadrant_label = {1: "HH", 2: "LH", 3: "LL", 4: "HL"}
sc_crime["lisa_cluster"] = sc_crime["lisa_q"].map(quadrant_label)
sc_crime.loc[sc_crime["lisa_p"] >= 0.05, "lisa_cluster"] = "Not significant"

sc_crime["lisa_cluster"].value_counts()

In [ ]:
# 5) Choropleth of the raw crime rate.
fig, ax = plt.subplots(figsize=(10, 8))
sc_crime.plot(
    column="rate_per_10k",
    cmap="YlOrRd",
    legend=True,
    legend_kwds={"label": "Crime rate per 10k residents"},
    edgecolor="white",
    linewidth=0.4,
    ax=ax,
)
ax.set_title("Total Crime Rate per 10,000 Residents by County (SC, 2024)")
ax.axis("off")
plt.savefig(OUTPUT_DIR / "crime_rate_per_10k_choropleth.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 6) Moran scatterplot (rate vs its spatial lag).
wy = lag_spatial(w, y)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y, wy, c="steelblue", s=60, alpha=0.8)

b, a = np.polyfit(y, wy, 1)
xs = np.array([y.min(), y.max()])
ax.plot(xs, a + b * xs, color="crimson", linestyle="--", label="Linear fit")

ax.axvline(y.mean(), color="gray", linestyle=":", linewidth=0.8)
ax.axhline(wy.mean(), color="gray", linestyle=":", linewidth=0.8)

ax.set_xlabel("Crime rate per 10k")
ax.set_ylabel("Spatial lag of crime rate per 10k")
ax.set_title(f"Moran Scatterplot (I = {moran.I:.3f})")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "moran_scatterplot.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 7) LISA cluster map.
cluster_colors = {
    "HH": "#d73027",  # high surrounded by high
    "LL": "#1a9641",  # low surrounded by low
    "HL": "#fdae61",  # high surrounded by low
    "LH": "#4575b4",  # low surrounded by high
    "Not significant": "#e0e0e0",
}

fig, ax = plt.subplots(figsize=(10, 8))
sc_crime.plot(
    color=sc_crime["lisa_cluster"].map(cluster_colors),
    edgecolor="white",
    linewidth=0.4,
    ax=ax,
)

handles = [mpatches.Patch(color=c, label=l) for l, c in cluster_colors.items()]
ax.legend(handles=handles, loc="lower right", title="LISA cluster (p < 0.05)")
ax.set_title("LISA Cluster Map of Crime Rate per 10k")
ax.axis("off")
plt.savefig(OUTPUT_DIR / "lisa_cluster_map.png", dpi=150, bbox_inches="tight")
plt.show()